# MES Feature Engineering

## Purpose

This notebook records the transition from the validated V2 event-time one-second tape to the frozen V1 one-minute foundation. The next research stage will create interpretable model features for market structure, price and momentum, volatility, participation, live-compatible inferred order flow, effort versus result, and session context.

The authoritative implementation is `src/feature_engineering_v1.py`. Notebook code uses that implementation for all executable aggregation and validation paths so a prototype cannot drift from production logic.

Production features must be reproducible from planned IBKR live data. Databento native aggressor-side fields are retained only as historical benchmark fields, not production model inputs.

## One-Minute Foundation: Timing and Aggregation Contract

A row at `decision_timestamp` `10:31:00` summarizes only the completed event-time interval `[10:30:00, 10:31:00)`. No contributing trade occurs at or after `10:31:00`. This is the historical event-time eligibility boundary that prevents look-ahead bias. A future live IBKR system must also wait until the final relevant event has been received and processed before acting: exchange event time and local receipt time are not literally identical.

The foundation retains OHLC, volume-weighted price sufficient statistics, activity and trade-size statistics, inferred and native benchmark order flow, session/contract identity, timestamps, and the contract-change marker. V2 contains only traded seconds, so `active_second_count` is the number of seconds in a minute that contained trades; it is not an imputed 60-second bar.

A minute may contain only one contract. Databento prices are unadjusted across rolls, so future returns, momentum, volatility, level distances, and all rolling price features must restart at each new contract.

`unique_price_levels` and `max_volume_at_price` are intentionally omitted. Summing one-second unique-price counts double-counts prices recurring across seconds; taking a maximum of one-second volume-at-price does not recover volume accumulated at that price through a minute. The minute layer deliberately avoids misleading approximations. V2 remains available if future research requires these fields; true minute/session volume-at-price needs a separate deliberate design.

In [ ]:
# Load one V2 session and use the authoritative, non-writing aggregation path.
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering_v1 import aggregate_seconds_to_minutes

V2_DIR = PROJECT_ROOT / 'data' / 'processed' / '1s' / 'full_history_v2_event_time'
MINUTE_OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / '1m' / 'full_history_v1'
sample_file = V2_DIR / 'MES_1s_2025-10-08.parquet'
seconds_df = pd.read_parquet(sample_file)
minute_df = aggregate_seconds_to_minutes(seconds_df)

print('Sample session:', sample_file.name)
print('1-second rows:', len(seconds_df))
print('1-minute rows:', len(minute_df))
print('Authoritative aggregation: PASS')
minute_df[['decision_timestamp', 'open', 'high', 'low', 'close', 'total_volume', 'trade_count', 'active_second_count', 'inferred_delta', 'contract']].head()

In [ ]:
# Manually check one aggregated minute against its source seconds.
seconds_for_check = seconds_df.copy()
seconds_for_check['timestamp_second'] = pd.to_datetime(seconds_for_check['timestamp_second'], utc=True)
seconds_for_check['minute_start'] = seconds_for_check['timestamp_second'].dt.floor('min')
first_minute = minute_df.iloc[0]
contributing_seconds = seconds_for_check.loc[
    seconds_for_check['minute_start'] == first_minute['minute_start']
].sort_values('timestamp_second')

assert first_minute['open'] == contributing_seconds['open'].iloc[0]
assert first_minute['high'] == contributing_seconds['high'].max()
assert first_minute['low'] == contributing_seconds['low'].min()
assert first_minute['close'] == contributing_seconds['close'].iloc[-1]
assert first_minute['total_volume'] == contributing_seconds['total_volume'].sum()
assert first_minute['trade_count'] == contributing_seconds['trade_count'].sum()
assert first_minute['inferred_delta'] == contributing_seconds['inferred_delta'].sum()
assert first_minute['active_second_count'] == len(contributing_seconds)
print('First-minute manual validation: PASS')

## Contract-Roll Protection

The continuous series has three preserved contract changes. The aggregation hard-fails if a minute would mix contracts. The `contract_change` marker identifies the first minute of the new contract, but later rolling price features must use contract-aware segments rather than treating the marker alone as a return adjustment.

In [ ]:
# Read-only inspection of a known roll boundary.
roll_seconds = pd.read_parquet(V2_DIR / 'MES_1s_2025-12-17.parquet')
roll_seconds['timestamp_second'] = pd.to_datetime(roll_seconds['timestamp_second'], utc=True)
roll_seconds['minute_start'] = roll_seconds['timestamp_second'].dt.floor('min')
contracts_per_minute = roll_seconds.groupby('minute_start')['instrument_id'].nunique()
assert (contracts_per_minute <= 1).all()
roll_rows = roll_seconds.loc[roll_seconds['contract_change'], ['timestamp_second', 'instrument_id', 'contract']]
print('No mixed-contract minute: PASS')
roll_rows

## Fixed Schema and Safe Writer

The source module defines the authoritative ordered 33-field Arrow schema. It exists so future pandas/PyArrow versions cannot silently alter physical Parquet types. The writer processes one session at a time, refuses overwrites, reloads its output, and is intentionally disabled below because the historical foundation is complete.

In [ ]:
# Historical disposable writer test: disabled so Run All never creates writer_test data.
import src.feature_engineering_v1 as feature_engineering_v1

RUN_ONE_MINUTE_WRITER_TEST = False

if RUN_ONE_MINUTE_WRITER_TEST:
    writer_test_dir = PROJECT_ROOT / 'data' / 'processed' / '1m' / 'writer_test'
    writer_result = feature_engineering_v1.write_one_minute_session(
        V2_DIR / 'MES_1s_2025-10-08.parquet',
        writer_test_dir,
    )
    print(writer_result)
else:
    print('Disposable one-minute writer test skipped; no files were written.')

In [ ]:
# Read-only preflight. The completed foundation must not be rebuilt.
source_files = sorted(V2_DIR.glob('MES_1s_*.parquet'))
minute_files = sorted(MINUTE_OUTPUT_DIR.glob('MES_1m_*.parquet'))
if len(source_files) != 242:
    raise ValueError(f'Expected 242 V2 source files, found {len(source_files)}.')
if len(minute_files) != 242:
    raise ValueError(f'Expected 242 frozen minute files, found {len(minute_files)}.')
print('Frozen source and minute file counts: PASS')

In [ ]:
# The completed production build is permanently disabled for normal notebook use.
RUN_FULL_HISTORY_1M = False

if RUN_FULL_HISTORY_1M:
    full_minute_results, full_minute_summary = feature_engineering_v1.build_full_history_minutes(
        V2_DIR,
        MINUTE_OUTPUT_DIR,
    )
    print(full_minute_summary)
else:
    print('Full-history minute build disabled; frozen output will not be modified.')

## Read-Only Full-History Audit

This audit reads one V2/minute session pair at a time. It validates the fixed Arrow and pandas dtype contracts, exact source-to-minute aggregation rules, leakage safety, per-minute order-flow identities, file/session coverage, known roll locations, and full-history totals. It never writes or modifies project data.

In [ ]:
from pathlib import Path

import pyarrow.parquet as pq

from src.feature_engineering_v1 import ONE_MINUTE_ARROW_SCHEMA, ONE_MINUTE_COLUMNS

EXPECTED_PANDAS_DTYPES = {
    'decision_timestamp': 'datetime64[ns, UTC]',
    'minute_start': 'datetime64[ns, UTC]',
    'session_date': 'object',
    'instrument_id': 'uint32',
    'contract': 'string',
    'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64',
    'price_volume_sum': 'float64', 'price_squared_volume_sum': 'float64',
    'total_volume': 'uint64', 'trade_count': 'uint32', 'max_trade_size': 'uint32',
    'size_squared_sum': 'uint64', 'active_second_count': 'int64',
    'uptick_count': 'uint32', 'downtick_count': 'uint32', 'same_price_count': 'uint32',
    'inferred_buy_volume': 'uint64', 'inferred_sell_volume': 'uint64',
    'inferred_unknown_volume': 'uint64',
    'inferred_buy_trade_count': 'uint32', 'inferred_sell_trade_count': 'uint32',
    'inferred_unknown_trade_count': 'uint32', 'inferred_delta': 'int64',
    'native_buy_volume': 'uint64', 'native_sell_volume': 'uint64',
    'native_unknown_volume': 'uint64', 'native_delta': 'int64',
    'first_trade_timestamp': 'datetime64[ns, UTC]',
    'last_trade_timestamp': 'datetime64[ns, UTC]',
    'contract_change': 'bool',
}
EXPECTED_ROLLS = [
    ('2025-12-17 00:00:00+00:00', 42003800, 'MESH6'),
    ('2026-03-18 00:00:00+00:00', 42005163, 'MESM6'),
    ('2026-06-17 00:00:00+00:00', 42003239, 'MESU6'),
]

def require(condition, session_date, message):
    if not condition:
        raise ValueError(f'{session_date}: {message}')

source_files = sorted(V2_DIR.glob('MES_1s_*.parquet'))
minute_files = sorted(MINUTE_OUTPUT_DIR.glob('MES_1m_*.parquet'))
expected_names = {'MES_1m_' + path.stem.removeprefix('MES_1s_') + '.parquet' for path in source_files}
require(len(source_files) == 242, 'full_history', 'expected 242 V2 source files')
require(len(minute_files) == 242, 'full_history', 'expected 242 minute files')
require({path.name for path in minute_files} == expected_names, 'full_history', 'source/output filename coverage mismatch')

audit_rows = []
observed_rolls = []
for file_number, source_path in enumerate(source_files, start=1):
    session_date = source_path.stem.removeprefix('MES_1s_')
    minute_path = MINUTE_OUTPUT_DIR / f'MES_1m_{session_date}.parquet'
    require(minute_path.exists(), session_date, 'missing minute output')
    require(
        pq.ParquetFile(minute_path).schema_arrow.remove_metadata() == ONE_MINUTE_ARROW_SCHEMA,
        session_date,
        'Arrow schema differs from the permanent 33-field contract',
    )

    # Keep memory bounded: one source/output session pair at a time.
    seconds = pd.read_parquet(source_path)
    minutes = pd.read_parquet(minute_path)
    require(list(minutes.columns) == ONE_MINUTE_COLUMNS, session_date, 'column order differs from contract')
    require(
        {column: str(dtype) for column, dtype in minutes.dtypes.items()} == EXPECTED_PANDAS_DTYPES,
        session_date,
        'pandas dtypes differ from contract',
    )
    for column in ['timestamp_second', 'first_trade_timestamp', 'last_trade_timestamp']:
        seconds[column] = pd.to_datetime(seconds[column], utc=True)
    seconds = seconds.sort_values('timestamp_second').reset_index(drop=True)
    seconds['_minute'] = seconds['timestamp_second'].dt.floor('min')
    for column in ['decision_timestamp', 'minute_start', 'first_trade_timestamp', 'last_trade_timestamp']:
        minutes[column] = pd.to_datetime(minutes[column], utc=True)

    grouped = seconds.groupby('_minute', sort=True)
    expected_starts = pd.Series(grouped.size().index).reset_index(drop=True)
    require(grouped['instrument_id'].nunique().le(1).all(), session_date, 'source minute mixes contracts')
    require(minutes['session_date'].astype(str).nunique() == 1, session_date, 'multiple session dates in minute file')
    require((minutes['session_date'].astype(str) == session_date).all(), session_date, 'session identity mismatch')
    require(minutes['decision_timestamp'].is_monotonic_increasing, session_date, 'decision timestamps are not monotonic')
    require(not minutes['decision_timestamp'].duplicated().any(), session_date, 'duplicate decision timestamp')
    require((minutes['decision_timestamp'] == minutes['minute_start'] + pd.Timedelta(minutes=1)).all(), session_date, 'decision timestamp rule failed')
    require((minutes['last_trade_timestamp'] < minutes['decision_timestamp']).all(), session_date, 'trade at or after decision timestamp')
    require(minutes['active_second_count'].between(1, 60).all(), session_date, 'active_second_count outside [1, 60]')
    require(len(minutes) == len(expected_starts), session_date, 'minute-row count differs from source')
    require(minutes['minute_start'].reset_index(drop=True).equals(expected_starts), session_date, 'minute starts differ from source')

    expected = {
        'instrument_id': grouped['instrument_id'].first(), 'contract': grouped['contract'].first(),
        'open': grouped['open'].first(), 'high': grouped['high'].max(),
        'low': grouped['low'].min(), 'close': grouped['close'].last(),
        'price_volume_sum': grouped['price_volume_sum'].sum(),
        'price_squared_volume_sum': grouped['price_squared_volume_sum'].sum(),
        'total_volume': grouped['total_volume'].sum(), 'trade_count': grouped['trade_count'].sum(),
        'max_trade_size': grouped['max_trade_size'].max(),
        'size_squared_sum': grouped['size_squared_sum'].sum(),
        'active_second_count': grouped['timestamp_second'].count(),
        'uptick_count': grouped['uptick_count'].sum(), 'downtick_count': grouped['downtick_count'].sum(),
        'same_price_count': grouped['same_price_count'].sum(),
        'inferred_buy_volume': grouped['inferred_buy_volume'].sum(),
        'inferred_sell_volume': grouped['inferred_sell_volume'].sum(),
        'inferred_unknown_volume': grouped['inferred_unknown_volume'].sum(),
        'inferred_buy_trade_count': grouped['inferred_buy_trade_count'].sum(),
        'inferred_sell_trade_count': grouped['inferred_sell_trade_count'].sum(),
        'inferred_unknown_trade_count': grouped['inferred_unknown_trade_count'].sum(),
        'inferred_delta': grouped['inferred_delta'].sum(),
        'native_buy_volume': grouped['native_buy_volume'].sum(),
        'native_sell_volume': grouped['native_sell_volume'].sum(),
        'native_unknown_volume': grouped['native_unknown_volume'].sum(),
        'native_delta': grouped['native_delta'].sum(),
        'first_trade_timestamp': grouped['first_trade_timestamp'].min(),
        'last_trade_timestamp': grouped['last_trade_timestamp'].max(),
        'contract_change': grouped['contract_change'].max(),
    }
    for field, expected_values in expected.items():
        require(
            minutes[field].reset_index(drop=True).equals(expected_values.reset_index(drop=True)),
            session_date,
            f'{field} aggregation differs from source seconds',
        )

    signed = lambda column: minutes[column].astype('int64')
    require((signed('inferred_buy_volume') + signed('inferred_sell_volume') + signed('inferred_unknown_volume') == signed('total_volume')).all(), session_date, 'inferred volume does not reconcile')
    require((signed('native_buy_volume') + signed('native_sell_volume') + signed('native_unknown_volume') == signed('total_volume')).all(), session_date, 'native volume does not reconcile')
    require((signed('inferred_buy_volume') - signed('inferred_sell_volume') == signed('inferred_delta')).all(), session_date, 'inferred delta does not reconcile')
    require((signed('native_buy_volume') - signed('native_sell_volume') == signed('native_delta')).all(), session_date, 'native delta does not reconcile')
    require((signed('inferred_buy_trade_count') + signed('inferred_sell_trade_count') + signed('inferred_unknown_trade_count') == signed('trade_count')).all(), session_date, 'inferred trade counts do not reconcile')

    observed_rolls.extend((str(row.minute_start), int(row.instrument_id), str(row.contract)) for row in minutes.loc[minutes['contract_change'], ['minute_start', 'instrument_id', 'contract']].itertuples(index=False))
    audit_rows.append({'second_rows': len(seconds), 'minute_rows': len(minutes), 'volume': int(minutes['total_volume'].sum()), 'trades': int(minutes['trade_count'].sum()), 'contract_changes': int(minutes['contract_change'].sum())})
    if file_number == 1 or file_number % 25 == 0 or file_number == len(source_files):
        print(f'Audited {file_number}/{len(source_files)} sessions')

audit_df = pd.DataFrame(audit_rows)
require(len(audit_df) == 242, 'full_history', 'session count mismatch')
require(int(audit_df['second_rows'].sum()) == 13_165_975, 'full_history', 'source-second total mismatch')
require(int(audit_df['minute_rows'].sum()) == 329_337, 'full_history', 'minute-row total mismatch')
require(int(audit_df['volume'].sum()) == 298_546_252, 'full_history', 'volume total mismatch')
require(int(audit_df['trades'].sum()) == 99_319_450, 'full_history', 'trade total mismatch')
require(int(audit_df['contract_changes'].sum()) == 3, 'full_history', 'contract-change total mismatch')
require(observed_rolls == EXPECTED_ROLLS, 'full_history', f'known rolls differ: {observed_rolls}')

print('Full-history one-minute audit: PASS')
print('Sessions:', len(audit_df))
print('Source one-second rows:', int(audit_df['second_rows'].sum()))
print('One-minute rows:', int(audit_df['minute_rows'].sum()))
print('Volume:', int(audit_df['volume'].sum()))
print('Trades:', int(audit_df['trades'].sum()))
print('Known contract changes:', observed_rolls)

## V1 One-Minute Foundation — Completed

The permanent V1 minute derivation from frozen V2 event-time data is complete and independently audited: 242 sessions, 13,165,975 source one-second rows, 329,337 minute rows, 298,546,252 volume, 99,319,450 trades, and three preserved contract changes.

`full_history_v1` here means version 1 of the minute derivation; it does not mean the disallowed receive-time 1-second V1 dataset. The run manifest records provenance and policy. The next step is model-feature creation from this frozen minute foundation.